# 01 - EDA & Preprocessing
Eksplorasi data dan pembuatan cache gambar (resize ke 224x224, grayscale JPEG).

In [ ]:
import os, sys, subprocess
REPO_URL = "azizahadilaa/document-image-classification-kaggle.git"  # <-- GANTI
REPO_DIR = "/kaggle/working/document-image-classification-kaggle"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.append(REPO_DIR)

import pandas as pd, numpy as np, torch
from scripts.utils import set_seed, find_data_dir
set_seed(42)
DATA_DIR = find_data_dir()
CACHE_DIR = "/kaggle/temp/cache"
IMG_SIZE = 224
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(DATA_DIR, train_df.shape, test_df.shape)

## Distribusi kelas

In [ ]:
import matplotlib.pyplot as plt
counts = train_df["class_name"].value_counts()
plt.figure(figsize=(10, 4)); counts.plot(kind="bar"); plt.title("Distribusi kelas (train)")
plt.tight_layout(); os.makedirs(f"{REPO_DIR}/reports/figures", exist_ok=True)
plt.savefig(f"{REPO_DIR}/reports/figures/class_distribution.png", dpi=150); plt.show()
print(counts)

## Contoh gambar tiap kelas

In [ ]:
from PIL import Image
classes = train_df.sort_values("label")["class_name"].unique()
fig, axes = plt.subplots(4, 4, figsize=(14, 16))
for ax, c in zip(axes.ravel(), classes):
    row = train_df[train_df.class_name == c].iloc[0]
    ax.imshow(Image.open(f"{DATA_DIR}/{row.filename}").convert("L"), cmap="gray"); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{REPO_DIR}/reports/figures/sample_images.png", dpi=120); plt.show()

## Cek ukuran gambar asli (sampel)

In [ ]:
sizes = [Image.open(f"{DATA_DIR}/{f}").size for f in train_df.filename.sample(200, random_state=0)]
print(pd.DataFrame(sizes, columns=["w", "h"]).describe())

## Cache gambar (sekali jalan, ~10-20 menit)

In [ ]:
from scripts.preprocess import cache_images
cache_images(train_df, DATA_DIR, CACHE_DIR, IMG_SIZE)
cache_images(test_df, DATA_DIR, CACHE_DIR, IMG_SIZE)
print(len(os.listdir(CACHE_DIR)), "gambar ter-cache")